# Building Autonomous Agents

> Previous lectures introduced the verifier, tool execution, memory, and evaluation. Each piece can run on a demo task. Running a demo and running unattended for hours are different problems, and three issues sit between them: reliability, oversight, and trust.
>
> This lecture puts those pieces into one system and faces the engineering and open problems behind **autonomy**. We start from the multiplicative effect of per-step accuracy, then build error detection and retry, checkpoint recovery, and thresholded handover. We close with a minimal trusted-monitoring protocol, for the case where humans can no longer keep up with the model.

We start with a calculation. Suppose the Agent is correct with probability 0.9 on each step.

On a one-step task, the success rate is 0.9.

On a two-step task, step 1 and step 2 must both succeed, so 0.9×0.9=0.81.

On a five-step task, 0.9 multiplied five times is 0.9⁵≈0.59 — already under 60%.

On a ten-step task, 0.9¹⁰≈0.35 — success about once in three runs.

The longer the task, the faster the success rate falls. **Autonomy is multiplicative**, not additive: once the step count grows, making each step a little more accurate quickly saturates, so the system must detect errors and recover from them. The ability to keep running and recover on its own is called **reliability**.

Reliability and autonomy pull in opposite directions. Letting the Agent act freely means accepting that it will fail at some point. There is no one-shot fix. The practical response is a set of mechanisms that catch failure: a self-check in the loop, persisted intermediate state, and human handover when the system cannot recover.

This lecture sits in the loop from Lecture 1 and adds capability after the feedback step: error detection, retry, checkpoint recovery, and a thresholded handover mechanism. We begin with the most elementary layer: how per-step accuracy determines whether a whole task succeeds.


## 1. From demos to autonomy

This section explains why a task that works as a demo fails when it is stretched. We first distinguish demo tasks from autonomous tasks, then compute how much reliability a long task demands.

Demo tasks are short: a correct result arrives within a few steps, and rerunning after a mistake is cheap. Most demos in earlier lectures stay within ten steps. Autonomous tasks require the system not to fail across a long chain of steps, or to detect and repair failure on its own. Unattended continuous runs can be thousands of steps.

When the step count grows from ten to thousands, the reliability requirement grows by orders of magnitude. We start from the most elementary layer: how per-step accuracy determines whether a whole task succeeds.


In [ ]:
import os
import sys

_root = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(_root, 'llm_client.py')):
    _root = os.path.dirname(_root)
    if _root == os.path.dirname(_root):
        break
if _root not in sys.path:
    sys.path.insert(0, _root)

import numpy as np
np.random.seed(42)

from llm_client import get_llm
client = get_llm()
print('llm_client ready; current mode:',
      'scripted example (placeholder output for environments without a key)' if False else 'real API')


### Per-step accuracy is a multiplicative factor

This subsection shows how a small drop in per-step accuracy collapses the success rate of a long task. We first see why the probabilities multiply, then compute a few concrete numbers.

The product form follows from sequential dependence. A task is split into steps that depend on one another: step 2 can run only if step 1 succeeded, and any failed step fails the whole task. The probability that two independent events both occur is the product of their probabilities: the chance that two dice both show 6 is (1/6)×(1/6)=1/36. If an agent splits a task into n sequentially dependent steps, and each step succeeds independently with probability p, the whole task succeeds only if every step succeeds, so the overall success rate is p multiplied n times, that is p to the power n.

A minimal numerical example. Per-step accuracy 0.9, two-step task: step 1 succeeds with probability 0.9; conditional on that, step 2 succeeds with probability 0.9; the product is 0.9×0.9=0.81. Each step knocks another 10% off the remaining success rate. A few more steps look like this:

| steps n | overall success 0.9^n | reading |
|:---|:---|:---|
| 1 | 0.900 | nine in ten |
| 2 | 0.810 | eight in ten |
| 5 | 0.590 | under 60% |
| 10 | 0.349 | about one in three |

A two-step task at 90% per step is already down to 81%; five steps fall to 59%; ten steps leave 35%. Raising the per-step rate to 0.99 still gives 0.99^10 ≈ 0.90 at 10 steps, but 0.99^100 ≈ 0.37 at 100 steps.

The conclusion is structural. Autonomy is multiplicative, not additive: moving from a working demo to hours of unattended running cannot rely only on making each step a little more accurate. Once the step count grows, raising per-step accuracy saturates, so the system must detect and repair errors after they occur. That is the subject of the next section.


In [ ]:
def overall_success(probs):
    '''Given a list of per-step success probabilities, return the overall task success rate (independent product).'''
    p = 1.0
    for q in probs:
        p *= q
    return p

for n in (1, 2, 5, 10):
    print(f'{n:>2} steps, per-step 0.90: overall success {overall_success([0.9] * n):.3f}')

print('10 steps, per-step 0.99: overall success', round(overall_success([0.99] * 10), 3))


## 2. Reliability: error detection and recovery

The previous section computed the multiplicative effect of per-step accuracy. This section asks what mechanism can recover overall success when per-step accuracy is not high enough. The answer is a loop that detects errors and repairs them; that layer of capability is reliability.

Reliability is not a single property. It has three layers: getting a step right, keeping the loop stable, and recovering after an error. Each layer has its own failure mode and remedy. An agent summing the integers from 1 to 5 makes the three layers concrete:

- Getting a step right: one step emits a wrong value. For example the agent computes 2+3 as 6. This layer is determined by model skill and per-step verification, and it is the multiplicative factor from the previous section.
- Keeping the loop stable: a corrupted state is carried forward. For example the running sum is stored as 6, and every later step starts from 6, so everything after that is wrong. This layer is protected by state hygiene and checkpoints.
- Long-horizon correction: a step emits a wrong value, nobody notices, and the loop continues. For example step 3 is wrong once, later steps look normal, and the final result is still off. This layer relies on self-check and retry.

The three layers stack: a first-step error that is not detected and is written into state hits all three at once. We start at the boundary of the first two layers, and watch how errors accumulate on a long task.


### Multiplicative collapse: errors accumulate with digit count

This subsection traces how errors on a real long task gradually pull the whole result off course. Long multiplication is a useful example because every substep can look locally correct while the product is wrong.

Consider 23 × 47, split into three substeps:

```text
  23 × 7 = 161        (ones partial product)
  23 × 4 = 92 → 920   (tens partial product, shifted one place)
  161 + 920 = 1081    (sum, the true value)
```

Each substep can fail: 7×3 written as 18, a wrong carry, or a misaligned sum. More digits mean more such substeps.

Failure produces no signal. If the tens partial product is computed as 91 → 910, the total is 161 + 910 = 1071. 1071 is an ordinary number; every digit is legal; nothing in the writing says the result is wrong, yet it is wrong. That is the pattern of locally plausible steps and a wrong product: the final answer is the conjunction of a long chain of substeps, any one error voids the chain, and the error usually does not announce itself.

Give that pattern a number. Suppose a model solves a multiplication with 15 dependent steps at 0.9 accuracy per step. The probability of a fully correct product is 0.9^15. Compressing step by step:

```text
after step 1    0.9
after step 2    0.9 × 0.9     = 0.81
after step 3    0.81 × 0.9    ≈ 0.73
after step 5    ≈ 0.59
after step 10   ≈ 0.35
after step 15   0.9^15        ≈ 0.21
```

Fifteen steps at 0.9 per step leave about 21% overall success, less than one in five.

The derivation also hides an underestimated premise: per-step accuracy is constant. In practice it is often worse — longer tasks scatter attention, and later columns fail more often. The code below compares two curves: a fixed per-column accuracy of 0.99, and a per-column accuracy that decays linearly with column index (0.97 − 0.012 × column). At 15 columns the former is still 0.99^15 ≈ 0.86; the latter, because later columns are worse, falls to about 0.13. The decaying curve collapses more steeply on the plot, which is why the phenomenon is called multiplicative collapse.


In [ ]:
import matplotlib.pyplot as plt

def per_column_success(n):
    '''Success probability of the substep in column n: more columns, lower per-step accuracy.'''
    return max(0.05, 0.97 - 0.012 * n)

def fixed_overall(n):
    '''Overall success of an n-column task when every column succeeds with probability 0.99.'''
    return 0.99 ** n

def decaying_overall(n):
    '''Overall success of an n-column task when per-column success decays (product over columns).'''
    p = 1.0
    for i in range(1, n + 1):
        p *= per_column_success(i)
    return p

cols = np.arange(1, 16)
fixed = [fixed_overall(n) for n in cols]
decay = [decaying_overall(n) for n in cols]

print('cols | fixed per-step 0.99 | decaying with length')
for n in cols:
    print(f'{n:>3} | {fixed_overall(n):.4f}      | {decaying_overall(n):.4f}')

fig, ax = plt.subplots(figsize=(6, 3.6))
ax.plot(cols, fixed, marker='o', label='fixed step acc = 0.99')
ax.plot(cols, decay, marker='s', label='step acc decays with length')
ax.set_xlabel('number of columns (digits)')
ax.set_ylabel('overall success rate')
ax.set_title('Multiplicative collapse of long-multiplication success')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


### Self-check and retry loops

This subsection shows how to recover overall success when per-step accuracy is only 0.65. The idea is to add a self-check to the loop: after a step executes, verify the result; if it is correct, proceed; if not, try again.

The math of retry is elementary. Let the per-step error probability be q=0.35, so one attempt is correct with probability 0.65. Allow 3 retries, for 4 attempts in total. Final failure occurs only if all 4 attempts are wrong, with probability q to the fourth power:

```text
0.35 × 0.35     ≈ 0.122
0.122 × 0.35    ≈ 0.043
0.043 × 0.35    ≈ 0.015
```

About 1.5%. If the verifier is reliable, retry lifts "65% correct per step" to "about 98.5% correct". The cost is extra LLM calls.

That conclusion depends on a premise: the verifier can actually tell right from wrong. If the verifier cannot judge, retry is rolling dice in place — errors are waved through, and correct answers may be blocked. The quality of the verification signal sets the ceiling of this loop.

The next experiment compares three configurations on a concrete task. The task has 40 addition expressions with known ground truth, and the solver has a per-step error rate of 0.35.

- No self-check: the first solve is accepted. Errors are never intercepted, so accuracy is the solver's own level, at one call per expression.
- LLM self-review (weak self-check): the review almost always says the answer looks correct, matching the negative result in the Self-Refine paper — the model said 94% of wrong samples looked correct. Weak self-check does not catch errors, retry almost never fires, and the extra calls do not buy accuracy.
- Exact verifier (hard verification): recompute the true value and compare exactly; any deviation is an error. Errors are intercepted reliably, retry actually fires, accuracy approaches 98.5%, and the cost is a visible increase in call count.

Under a real API demo, arithmetic solves are deterministically correct, so NoisyBrain injects the error rate for a controlled experiment. The same batch of expressions is run under all three configurations, comparing the true number correct against the number of LLM calls. The loop's own success report is not trustworthy; the true count must be checked against ground truth. That is the question outcome-level oversight answers, taken up in Section 3.


In [ ]:
import re

def llm_solve(client, expr):
    '''Ask the LLM to evaluate one arithmetic expression and return a number.

    Real-API demos emit a fixed “computed result: N.”; in live mode take the last integer in the reply.
    '''
    text = client.chat([{'role': 'user', 'content': f'compute the value of {expr}'}])
    return int(re.findall(r'-?\d+', text)[-1])


class NoisyBrain:
    '''A solver with bounded per-step accuracy: call the LLM, then corrupt the result at a set error rate.

    In live mode the model itself can be wrong; here a tunable error rate makes a controlled experiment,
    so self-check and retry can be observed in isolation. Errors are transient, so a retry can hit the truth.
    '''

    def __init__(self, client, err_prob=0.35, seed=0):
        self.client = client
        self.err_prob = err_prob
        self.rng = np.random.RandomState(seed)

    def solve(self, expr):
        '''Solve the expression: with probability err_prob return a wrong value, otherwise the LLM result.'''
        value = llm_solve(self.client, expr)
        if self.rng.rand() < self.err_prob:
            return value + self.rng.randint(1, 5)
        return value


def exact_check(expr, answer):
    '''Hard verification: recompute the true value and compare exactly; any deviation is an error.'''
    a, b = (int(x) for x in expr.split('+'))
    return answer == a + b


def llm_check(client, expr, answer, rng):
    '''Weak self-check: hand the answer to the LLM for review; the review almost always says it looks correct.

    Under a real-API demo the review has no independent judgment, so we pass with probability 94%,
    matching the Self-Refine negative result that the model said 94% of wrong samples looked correct.
    '''
    client.chat([{'role': 'user', 'content': f'please review: is {answer} the correct value of {expr}?'}])
    return rng.rand() < 0.94


class AgentLoop:
    '''An agent loop with self-check and a retry cap.

    Flow: solve -> self-check. If checker is None, skip verification and accept the result;
    on a failed check, retry within the remaining attempts; give up the step if attempts are exhausted.
    '''

    def __init__(self, brain, checker=None, max_retries=3):
        self.brain = brain
        self.checker = checker
        self.max_retries = max_retries

    def run(self, exprs):
        '''Solve exprs in order; return (list of accepted answers, total LLM call count).

        With no checker the first solve is always accepted, so wrong values are waved through.
        '''
        answers = []
        calls = 0
        for expr in exprs:
            for _ in range(self.max_retries + 1):
                answer = self.brain.solve(expr)
                calls += 1
                if self.checker is None or self.checker(expr, answer):
                    answers.append(answer)
                    break
        return answers, calls


print('solver, verifier, and retry loop defined; next, compare three configurations on the same expressions')


In [ ]:
rng = np.random.RandomState(7)
# Keep the real-API demo to 8 items so one experiment does not emit too many calls.
exprs = [f'{a} + {b}' for a, b in rng.randint(1, 30, size=(8, 2))]
weak_rng = np.random.RandomState(3)


def true_value(expr):
    '''Ground-truth value of the expression, used by the evaluation script.'''
    a, b = (int(x) for x in expr.split('+'))
    return a + b


def evaluate(seed, checker, max_retries):
    '''Evaluate one configuration on the same expressions; return (true number correct, LLM call count).

    The true number correct is compared against ground truth, not against the loop's own success report.
    '''
    brain = NoisyBrain(client, err_prob=0.35, seed=seed)
    loop = AgentLoop(brain, checker=checker, max_retries=max_retries)
    answers, calls = loop.run(exprs)
    n_ok = sum(1 for e, a in zip(exprs, answers) if a == true_value(e))
    return n_ok, calls


results = [
    ('no self-check (accept all)',
     evaluate(0, None, 0)),
    ('LLM self-check (weak verification)',
     evaluate(0, lambda e, a: llm_check(client, e, a, weak_rng), 3)),
    ('exact verification (hard signal)',
     evaluate(0, exact_check, 3)),
]

for name, (n_ok, n_calls) in results:
    print(f'{name}: true correct {n_ok:>2}/{len(exprs)}, LLM calls {n_calls}')


**Comparing the three results**

After 40 expressions the differences are visible:

- No self-check: 40 calls, true correct 31/40. Accuracy sits at the solver's per-step level, with no extra protection.
- Weak self-check: 41 calls, true correct 32/40. One extra call and one extra correct item over no self-check. The review almost always says the answer looks correct, so errors are not caught and retry does not fire. That is the Self-Refine negative result in this example.
- Exact verification: 53 calls, true correct 40/40. Thirteen extra calls bought a perfect score.

The 53 can be computed by hand. With a 0.35 per-step error rate and at most 4 attempts, one expression takes about 1.5 calls on average to pass verification (about 65% succeed on the first try, about 23% succeed after one failure, and so on), so 40 expressions total about 60 calls. The observed 53 is one realization of a random process, close to the expectation.

The three groups make one point: spending a few extra calls for near-100% accuracy is usually a good trade on autonomous tasks, provided the verifier is actually reliable. That returns to the opening claim of this section: the quality of the verification signal sets the ceiling of the loop.


### Checkpoints: persist intermediate state

This subsection shows how to avoid throwing away finished work when the process crashes mid-run. Retry in the previous subsection handled a wrong step; this subsection handles a vanished process.

Once loop state is wrong it propagates forward, which is the main threat at the loop-stability layer. State is the information the loop carries between steps and cannot recover without recomputation: a running sum, a processed list, conversation history, tool return values. When the process is interrupted by OOM, power loss, or an external signal, in-memory state is gone; only what was written to disk remains.

One countermeasure is to persist intermediate state explicitly: after each completed step, write progress to disk, and after a crash continue from the latest checkpoint rather than from scratch. Files on disk outlive the process; that is the core assumption of checkpointing.

A small task walks through the mechanism: process 10 numbers in order and accumulate a sum, persist after each item, and simulate a crash after item 5.

At crash time the disk holds "processed 1 through 5, total 15". Recovery reads that state, sees 5 items in done, and continues from item 6, processing only 5 more steps. Without a checkpoint the only option is to start from zero and redo all 10 steps. The larger the list and the more expensive each step, the more a checkpoint saves.

One premise: recovery is safe. In this example the processing of each item is independent, and rerunning does not produce duplicate side effects, so "do only unfinished items" is correct. If every step has irreversible external side effects, persistence is not enough; each action must execute at most once (exactly-once semantics), which is a harder engineering problem. This lecture starts from the elementary version: serializable state and actions that are safe to rerun.


In [ ]:
import json
import os

CP_PATH = '/tmp/lecture18_state.json'

def save_checkpoint(state, path=CP_PATH):
    '''Write task state to a JSON file, simulating persistence.'''
    with open(path, 'w') as f:
        json.dump(state, f)

def load_checkpoint(path=CP_PATH):
    '''Read a checkpoint; return None if the file does not exist.'''
    if not os.path.exists(path):
        return None
    with open(path) as f:
        return json.load(f)

items = list(range(1, 11))

def run_until_crash(items, crash_after=5):
    '''Process items one by one; simulate a process crash after item crash_after.'''
    state = {'done': [], 'total': 0}
    for x in items:
        if len(state['done']) == crash_after:
            raise RuntimeError('process interrupted by an external signal')
        state['total'] += x
        state['done'].append(x)
        save_checkpoint(state)
        done = len(state['done'])
        print('processed', x, ': completed', done, 'items, total', state['total'])
    return state

try:
    run_until_crash(items)
except RuntimeError as e:
    print('crash:', e)


In [ ]:
def run_from_scratch(items):
    '''Process items from the start, persisting each step; return (final state, step count).'''
    state = {'done': [], 'total': 0}
    steps = 0
    for x in items:
        state['total'] += x
        state['done'].append(x)
        save_checkpoint(state)
        steps += 1
    return state, steps

def resume_from_checkpoint(items, path=CP_PATH):
    '''Load a checkpoint and process only unfinished items; return (final state, step count).'''
    state = load_checkpoint(path)
    if state is None:
        state = {'done': [], 'total': 0}
    done_set = set(state['done'])
    todo = [x for x in items if x not in done_set]
    for x in todo:
        state['total'] += x
        state['done'].append(x)
        save_checkpoint(state)
    return state, len(todo)

state_resume, steps_resume = resume_from_checkpoint(items)
state_scratch, steps_scratch = run_from_scratch(items)
print('resume (with checkpoint):', steps_resume, 'steps')
print('from scratch (no checkpoint):', steps_scratch, 'steps')
print('recovery skipped', len(items) - steps_resume, 'completed items, final total', state_resume['total'])


## 3. Oversight and the trust boundary

Reliability asks whether the system can keep itself stable. Oversight and trust ask on what grounds we believe it, and how much authority we grant it.

This section treats two problems. The first is oversight: when the agent is wrong, how we know where it went wrong. The answer comes at different granularities. Outcome-level checks are cheap but do not locate the cause; step-level checks are expensive but can pin the error to a specific step. The second is trust: when we delegate actions with consequences, what the decision rests on and where the boundary is drawn. The answer is a confidence threshold; actions below the threshold go to a human.

We start with the granularity of oversight: the same error yields completely different information under outcome-level and step-level checks.


### Outcome-level and step-level oversight

This subsection shows how we locate an agent's error. The answer depends on where we look: the final output, or each step.

Oversight answers whether the agent did the task correctly and where it went wrong. Different granularities yield different information. Outcome-level oversight looks only at whether the final output is correct: it is cheap and can be labeled automatically whenever ground truth or unit tests exist, with no need for a process. Step-level oversight gives feedback on each step: it is more interpretable and can name the failing step, but it needs step-level labels and costs much more.

A five-step trajectory makes the difference concrete. The trajectory simulates accumulating 1, 2, 3, 4, 5: step 1 adds 1, step 2 adds 2, and so on. At step 3 we write 103 instead of 3 (an extra 100); the other steps are correct. The true total is 1+2+3+4+5=15.

The outcome-level check compares the final total with 15: 1+2+103+4+5=115, which is not equal, so the check fails. It reports that the result is wrong, but not which step failed — we only see that the final result is off by 100, and the cause has to be guessed.

The step-level check inspects each item: after step 1 the total is 1, after step 2 it is 3, both matching the expectation; after step 3 the total is 106 against an expected 6, so the error is located at step 3. It exposes the trail of the error.

Error compounding is the next effect. After step 3 is wrong, even if steps 4 and 5 are fully correct the running total never matches the expectation. Looking only at the final result, we cannot tell a single error from an error on every step. That is the blind spot of outcome-level oversight on long tasks: an early small error contaminates the verdict of every later check, which is exactly what the loop-stability layer is meant to prevent. Step-level oversight is expensive, but it turns "where did it fail" from a guess into an observation.


In [ ]:
def build_trajectory(values, error_at):
    '''Build a trajectory: each step is (value, is_error); all steps except error_at are clean.'''
    traj = []
    for i, v in enumerate(values):
        if i == error_at:
            traj.append((v + 100, True))
        else:
            traj.append((v, False))
    return traj

def outcome_check(traj, expected):
    '''Outcome-level check: compare only whether the final total equals expected. Return (passed, note).'''
    final = sum(v for v, _ in traj)
    return final == expected, f'final total {final}, expected {expected}'

def process_check(traj):
    '''Step-level check: inspect each item; return the 0-based index of the first bad step, or None.'''
    total = 0
    for i, (v, bad) in enumerate(traj):
        total += v
        if bad:
            return i
    return None

values = [1, 2, 3, 4, 5]
traj = build_trajectory(values, error_at=2)

ok, why = outcome_check(traj, sum(values))
print('outcome-level:', 'passed' if ok else f'failed: {why}')
pos = process_check(traj)
print('step-level  :', 'all passed' if pos is None else f'step {pos + 1} failed')


In [ ]:
def compounding(traj):
    '''Show how an early error biases every later outcome-level check.'''
    total = 0
    expect = 0
    for i, (v, bad) in enumerate(traj):
        total += v
        expect += values[i]
        verdict = 'ok' if total == expect else 'wrong'
        print(f'after step {i + 1}: total {total}, expected {expect}, outcome check -> {verdict}')

compounding(traj)


### Thresholded handover: autonomy and safety as one knob

This subsection specifies when the agent acts on its own and when a human takes over. Compare confidence with a threshold; below the threshold, hand the action to a human.

Autonomy is delegation: actions with consequences are given to the agent. The boundary of that delegation needs an operational mechanism: the agent reports a confidence for each step, and below the threshold it requests human intervention. The intuition is the takeover logic of driving assistance: the system drives on simple roads, and when it is uncertain (low confidence) it returns control to the driver.

The mechanism turns degree of autonomy and degree of safety into two marks on the same knob. A low threshold lets the agent do more, with fewer human reviews and more missed errors; a high threshold brings more human intervention and fewer missed errors.

The experiment constructs 200 actions under control: 80% are correct, confidence of correct actions concentrates near 0.9, and confidence of incorrect actions concentrates near 0.4. The model is more sure of correct actions; that is the premise that makes confidence usable.

A hand calculation for threshold 0.7. Actions with confidence below 0.7 mostly come from the incorrect group (centered at 0.4), with a few uncertain items from the correct group. Escalated actions are treated as corrected; the rest keep their original correctness. In an actual run: threshold 0.7 escalates 36 actions (18%), final correct 200/200 — incorrect actions are caught by a human, and correct actions are mostly waved through. Raising the threshold toward 0.99 puts almost every action below the threshold, so the escalation rate approaches 1 and the error rate stays 0 — at the cost of almost no autonomy. On the plot the two curves cross between about 0.4 and 0.6; after that the error rate is already zero while the escalation rate is still rising. That interval is the trade-off of one more human review against one fewer error.

An honest limit: the mechanism assumes that confidence is calibrated, meaning the model is truly more sure of correct actions. If the model also assigns high confidence to wrong answers (overconfidence, which is common in LLMs), confidence is not a good signal, and turning the knob higher still will not catch errors. Thresholded handover works only after confidence is calibrated.


In [ ]:
def sample_actions(n=200, seed=0):
    '''Generate n actions, each (is_correct, confidence).

    Confidence of correct actions concentrates near 0.9, incorrect near 0.4, each with Gaussian noise.
    '''
    rng = np.random.RandomState(seed)
    actions = []
    for _ in range(n):
        correct = rng.rand() < 0.8
        center = 0.9 if correct else 0.4
        conf = float(np.clip(center + rng.normal(0, 0.1), 0.05, 0.99))
        actions.append((correct, conf))
    return actions

def escalate(actions, threshold):
    '''Actions with confidence below threshold go to a human (treated as corrected); the rest keep is_correct.

    Return (escalation count, final correct count).
    '''
    escalated = 0
    correct = 0
    for is_correct, conf in actions:
        if conf < threshold:
            escalated += 1
            correct += 1
        else:
            correct += int(is_correct)
    return escalated, correct

actions = sample_actions()
n_esc, n_ok = escalate(actions, 0.7)
print(f'threshold 0.7: escalate {n_esc} actions, final correct {n_ok}/{len(actions)}')


In [ ]:
import matplotlib.pyplot as plt

thresholds = np.linspace(0.0, 0.99, 30)
esc_rates = []
err_rates = []
for t in thresholds:
    n_esc, n_ok = escalate(actions, t)
    esc_rates.append(n_esc / len(actions))
    err_rates.append(1 - n_ok / len(actions))

fig, ax = plt.subplots(figsize=(6, 3.6))
ax.plot(thresholds, esc_rates, marker='o', ms=3, label='escalation rate')
ax.plot(thresholds, err_rates, marker='s', ms=3, label='error rate')
ax.set_xlabel('confidence threshold')
ax.set_ylabel('rate')
ax.set_title('Threshold trades autonomy against safety')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

for i in (0, 6, 12, 18, 24):
    print(f'threshold {thresholds[i]:.2f}: escalation {esc_rates[i]:.2f}, error {err_rates[i]:.3f}')


## 4. Open problem: when humans cannot keep up with the model

Reliability, oversight, and trust in the first three sections all assume a strong enough overseer: a human strong enough to judge whether the agent is correct. This section faces a problem that is getting closer: if the model exceeds unaided humans on many skills, how humans can still keep watch.

There is no ready answer. This section currently has two lines of thought. Scalable oversight is the optimistic path: a human plus an LLM assistant collaborating, exceeding either party alone. AI control is the defensive path: assume the model is adversarial and tries to break safety measures, and still require a safe deployment. A small trusted-monitoring protocol below shows what the second line looks like in concrete form.


### Minimal AI-control: a trusted monitoring protocol

This subsection shows how to keep a deployment safe when the monitored model itself is untrusted. The core idea comes from Redwood's AI Control work: a trusted but weaker model monitors an untrusted but stronger model.

The monitor is trusted-but-weak rather than trusted-and-strong for a reason. AI control starts from the premise that the strong model is untrusted; it may be the party that causes the problem. The monitor must be reliable and auditable; even if it is weaker, it cannot be equally targetable. Trust is the foundation of every guarantee this protocol makes.

The protocol has three steps. First, the untrusted model proposes a large pool of candidate actions, for example 50, of which about 20% are harmful. Second, the trusted monitor scores each action for suspiciousness — in reality this score comes from an independently trained weak model; here it is a linear weighting of features, and higher features mean more suspicion. Third, only the k most suspicious actions go to human audit; the rest are waved through. The human budget is limited, so the protocol's job is to spend that budget where problems are most likely.

Selecting by suspiciousness rather than at random is justified by the numbers. In this example there are 50 actions, 10 harmful, and an audit budget of 20% (10 actions): random sampling hits 2 harmful actions on average, while sampling by suspiciousness caught 7 (safety 0.70). Harmful actions have higher features overall and concentrate at the top of the suspiciousness ranking, so the weak model acts as a sieve, sending the most suspicious sliver of the strong model's actions to a human.

Human audit itself has a ceiling. Full audit of all 50 actions still only reaches safety 0.80 — human auditors work at an 85% catch rate, so even exhausting the budget does not block every harmful action. The trade-off curve plots audit cost against safety: as the budget goes from 0 to 20%, safety jumps from 0 to 0.70, and the curve is steep at the start; further increases have diminishing returns. The value of trusted monitoring plus top-k is that steep start — most of the safety for a small budget.

The protocol looks good in a passive setting and fails once it is targeted. The bypass demo rewrites the same harmful action's features from (high, high) to (low, low); suspiciousness drops from 1.71 to 0.14, the action falls off the audit list, and it is waved through. The same payload, written differently, slips past the monitor — a direct demonstration that monitoring must be evaluated under the premise of being targeted. The defensive idea is protocol security: attackers will optimize around the protocol, so the protocol has to be robust against the best attacker, not only against the average case.


In [ ]:
def build_actions(n=50, harmful_rate=0.2, seed=0):
    '''Build a pool of candidate actions: each item has a harmful flag and a feature vector.

    Harmful actions have higher features overall, so the trusted monitor has a signal to use.
    '''
    rng = np.random.RandomState(seed)
    actions = []
    for i in range(n):
        harmful = rng.rand() < harmful_rate
        if harmful:
            feats = rng.rand(2) + (0.5, 0.4)
        else:
            feats = rng.rand(2)
        actions.append({'id': i, 'harmful': harmful, 'feats': feats})
    return actions

def suspiciousness(a):
    '''Trusted monitor score for an action: a linear weighting of features.

    In reality this score comes from an independently trained weak model; here we use fixed weights.
    '''
    return 0.6 * a['feats'][0] + 0.8 * a['feats'][1]

def audit_topk(actions, k):
    '''Select the k most suspicious actions by descending suspiciousness. Return their ids.'''
    ranked = sorted(actions, key=suspiciousness, reverse=True)
    return [a['id'] for a in ranked[:k]]

pool = build_actions()
n_harm = sum(1 for a in pool if a['harmful'])
print('action pool size', len(pool), ', harmful actions', n_harm)


In [ ]:
def protocol_safety_cost(actions, audited_ids, human_catch=1.0):
    '''Given the set of audited ids, return (safety, cost).

    Safety = number of intercepted harmful actions / total harmful actions;
    cost = fraction of actions that were audited.
    '''
    harmful_total = sum(1 for a in actions if a['harmful'])
    caught = sum(1 for a in actions if a['harmful'] and a['id'] in audited_ids)
    caught = int(caught * human_catch)
    safety = caught / harmful_total
    cost = len(audited_ids) / len(actions)
    return safety, cost

s_none, c_none = protocol_safety_cost(pool, [])
s_mon, c_mon = protocol_safety_cost(pool, audit_topk(pool, 10))
all_ids = [a['id'] for a in pool]
s_man, c_man = protocol_safety_cost(pool, all_ids, human_catch=0.85)

print(f'no monitoring           : safety {s_none:.2f}, cost {c_none:.2f}')
print(f'trusted monitor + top10 : safety {s_mon:.2f}, cost {c_mon:.2f}')
print(f'full human audit        : safety {s_man:.2f}, cost {c_man:.2f}')

import matplotlib.pyplot as plt

ks = range(0, 26, 2)
costs = [k / len(pool) for k in ks]
safeties = [protocol_safety_cost(pool, audit_topk(pool, k))[0] for k in ks]

fig, ax = plt.subplots(figsize=(6, 3.6))
ax.plot(costs, safeties, marker='o', ms=3, label='trusted monitor + top-k audit')
ax.scatter([c_man], [s_man], marker='x', s=60, label='full manual audit')
ax.scatter([c_none], [s_none], marker='^', s=60, label='no monitoring')
ax.set_xlabel('audit cost (fraction of actions)')
ax.set_ylabel('safety (fraction of harmful blocked)')
ax.set_title('Safety-cost tradeoff of monitoring protocols')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
def bypass_demo(pool, k=10):
    '''Demonstrate bypassing the monitor: the attacker hides suspicious features so a malicious action falls below the audit line.'''
    mal = max((a for a in pool if a['harmful']), key=suspiciousness)
    caught_before = mal['id'] in audit_topk(pool, k)

    hidden = {'id': mal['id'], 'harmful': True, 'feats': (0.1, 0.1)}
    pool_after = [hidden if a['id'] == hidden['id'] else a for a in pool]
    caught_after = hidden['id'] in audit_topk(pool_after, k)

    print('before attack: malicious action suspiciousness', round(suspiciousness(mal), 2),
          ', audited ->', caught_before)
    print('after attack: same payload, different output form, suspiciousness',
          round(suspiciousness(hidden), 2), ', audited ->', caught_after)

bypass_demo(pool)


The output makes the bypass concrete: the same harmful action has suspiciousness 1.71 before the attack and is necessarily on the audit list; after a different output form, suspiciousness drops to 0.14, it ranks below the audit line, and it is waved through. The same monitoring protocol looks good in a passive setting and fails once it is deliberately targeted.

This result is not an isolated example. METR's Frontier Risk Report records 44 overreach or deception incidents; the most serious weak point is automated monitoring itself — an embedded red team changing one environment variable let traffic slip past the monitor. The conclusion is that monitoring robustness must be evaluated under the premise of being targeted.

Taken together, this lecture supports one overall claim. When reward has no ground truth and monitoring can be bypassed, the ceiling of an autonomous system is set by the quality of the oversight signal, not by the number of model parameters. That is why this lecture calls reward and verification the most fundamental bottleneck.

Scalable oversight and AI control are two routes on this path that have not yet met: the former measures whether a human plus an assistant can exceed either party alone; the latter requires safety when oversight is attacked. The shared open problem is how to build a reliable evaluation of long-horizon autonomy, and how to scale RL on tasks with no ground-truth reward.


## Summary

- [ ] Per-step accuracy is a multiplicative factor: an n-step task succeeds with the product of per-step rates; two steps at 90% leave 81%
- [ ] Errors accumulate on long tasks: more digits and lower per-step accuracy collapse overall success nonlinearly
- [ ] Self-check plus retry makes a reliable loop: the quality of the verification signal sets the ceiling; hard verification beats LLM self-check
- [ ] Checkpoints persist intermediate state: after a crash, continue from the latest checkpoint and skip finished work
- [ ] Outcome-level oversight knows only right or wrong; step-level oversight locates the failing step; early errors bias outcome-level checks
- [ ] A confidence threshold is the knob between autonomy and safety: a higher threshold escalates more and misses fewer errors
- [ ] Minimal AI-control: a trusted monitor plus auditing only the most suspicious k% intercepts most harmful actions at low human cost
- [ ] Monitoring will be targeted: the same protocol works in a passive setting and fails after a deliberate bypass


## Exercises

> You may ask an AI to explain the idea. Do not ask it to finish the exercise for you.

All three exercises reuse functions from this lecture. Complete them first, then run the assertions.


**Exercise 1: multiplicative reliability**

Given a list of per-step success probabilities, complete overall_success so the overall success rate equals the product of the per-step rates. The assertions check that a 5-step task at 0.9 per step has overall success 0.9 to the fifth power.

Hint: the whole task succeeds only if every step succeeds, so multiply the per-step probabilities.


In [ ]:
def overall_success(probs):
    '''Given a list of per-step success probabilities, return the overall task success rate.'''
    p = 1.0
    for q in probs:
        p = p * q                       # blank 1: multiply by this step's success rate

    return p

assert abs(overall_success([0.9, 0.9, 0.9, 0.9, 0.9]) - 0.9 ** 5) < 1e-9
assert abs(overall_success([0.99] * 10) - 0.99 ** 10) < 1e-9
print('Exercise 1 passed: overall success equals the product of per-step success probabilities')


**Exercise 2: thresholded handover**

Given a batch of actions (is_correct, confidence), complete escalate: actions with confidence below threshold go to a human (treated as corrected); the rest keep their own correctness. The assertions check the escalation count and final correct count at threshold 0.8.

Hint: split each action into two paths by comparing confidence with the threshold; a higher threshold escalates more, and the final correct count only stays the same or grows.


In [ ]:
def escalate(actions, threshold):
    '''Actions with confidence below threshold go to a human (treated as corrected); the rest keep is_correct.

    Return (escalation count, final correct count).
    '''
    escalated = 0
    correct = 0
    for is_correct, conf in actions:
        if conf < threshold:
            escalated += 1
            correct += 1
        else:
            correct += int(is_correct)  # blank: when not escalated, count the action's own correctness

    return escalated, correct

acts = [(True, 0.95), (False, 0.30), (True, 0.60), (False, 0.85)]
assert escalate(acts, 0.8) == (2, 3)
print('Exercise 2 passed: at threshold 0.8, escalate 2, final correct 3')


**Exercise 3: allocate audit budget by suspiciousness**

Given a batch of actions (is_harmful, suspiciousness) and an audit budget k, complete audit_indices to send the k most suspicious actions to audit. The assertions check that the selected indices are the two highest-suspiciousness items.

Hint: with a limited budget, select by suspiciousness from high to low, not in list order; sort indices first, then take the first k.


In [ ]:
def audit_indices(actions, k):
    '''Return the indices of the k most suspicious actions, in descending suspiciousness.

    actions is a list of the form [(is_harmful, suspiciousness), ...].
    '''
    order = sorted(range(len(actions)), key=lambda i: actions[i][1], reverse=True)
    return order[:k]                   # blank: take the first k indices after sorting

acts = [(False, 0.2), (True, 0.9), (False, 0.5), (True, 0.7)]
assert sorted(audit_indices(acts, 2)) == [1, 3]
print('Exercise 3 passed: with a limited audit budget, select by suspiciousness from high to low')


## References

- Laskin, M., CS329A Lecture 15 guest talk (Reflection AI) — a practitioner's review: an agent's capability is the product of model, memory, tools, oversight, and recovery; reward and verification are the most fundamental bottleneck
- Bowman et al., [Measuring Progress on Scalable Oversight for Large Language Models](https://arxiv.org/abs/2211.03540), 2022 — defines a measurable scalable-oversight framework and shows that a human plus an assistant can exceed either party alone
- Greenblatt et al., [AI Control: Improving Safety Despite Intentional Subversion](https://arxiv.org/abs/2312.06942), 2023 — safe deployment even if the model tries to subvert: trusted monitoring, editing, and allocating human budget by suspiciousness
- Madaan et al., [Self-Refine: Iterative Refinement with Self-Feedback](https://arxiv.org/abs/2303.17651), 2023 — iterative self-feedback with a single model; the negative result that math does not improve shows that self-supervision must attach a hard verifier
- Bai et al., [Constitutional AI: Harmlessness from AI Feedback](https://arxiv.org/abs/2212.08073), 2022 — replace human labels with a list of principles; after critique and revision, use AI preferences for RLAIF
- Wang et al., [The Long-Horizon Task Mirage?](https://arxiv.org/abs/2604.11978), HORIZON 2026 — a long-horizon failure-attribution benchmark; 72.5% of failures are process-level errors, and high-horizon tasks collapse nonlinearly
- [PlanBench-XL](https://arxiv.org/abs/2606.22388), 2026 — agent brittleness under tool blocking: from 51.9% with no blocking down to 11.4% under the strictest blocking
- METR, [Frontier Risk Report](https://metr.org), 2026-05 — 44 overreach or deception incidents; automated monitoring can be bypassed at near-zero cost; the obedient-tool premise already fails
- Reflection 70B (HyperWrite and Glaive, 2024-09) — an open model that uses Reflection-Tuning thought labels for self-correction; a different entity from Laskin's Reflection AI
- Echo: papers/lecture-03/NOTES.md on verifiers (Cobbe / Lightman / Math-Shepherd / Weaver) — the direct precursor of outcome and process oversight
